# pc2beam — IFC → HELIOS++ simulation demo

This notebook walks through:

1. Tessellating IFC geometry and exporting **Wavefront OBJ** for HELIOS++ (default: **one OBJ per IFC instance** so LAS `hitObjectId` matches scene parts; optional merged mesh via `helios_one_obj_per_instance=False`).
2. Writing a **HELIOS++** multi-part scene + survey XML driven by `tools/helios_generation/config/scanners_example.yaml`.
3. Optionally running the **`helios`** CLI to produce a **LAS** point cloud, then visualizing the simulated scan **colored by instance id** when `hitObjectId` varies.

**Local (reproducible):** create the Conda env from `environment.yml` (pins **`helios`** and **`ifcopenshell`**). Stock HELIOS data is usually under `$CONDA_PREFIX/share/helios` or bundled as **`pyhelios`** in `site-packages`; `pc2beam` auto-resolves that. Set **`HELIOS_DATA_PATH`** only for a custom HELIOS data root (folder whose `data/` tree includes `platforms.xml`). See [HELIOS++](https://github.com/3dgeo-heidelberg/helios) upstream.

**Google Colab:** pip installs in this notebook do **not** include the `helios` binary — keep **`RUN_HELIOS = False`** unless you use a custom runtime with HELIOS++.

---
**Note:** the notebook detects Google Colab vs local execution, similar to `demo.ipynb`.

## environment detection and setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    print("environment: Google Colab")
except ImportError:
    IN_COLAB = False
    print("environment: local (expected Conda env: pc2beam, Python 3.12)")

if IN_COLAB:
    print("installing dependencies...")
    !pip install -q ifcopenshell plotly omegaconf numpy laspy
    !rm -rf /content/pc2beam
    !git clone https://github.com/fnoi/pc2beam.git
    import sys
    from pathlib import Path

    project_root = Path("/content/pc2beam")
    sys.path.insert(0, str(project_root))
    print("HELIOS++ is not installed on Colab by default; keep `RUN_HELIOS = False` or use a custom runtime.")
else:
    import sys
    from pathlib import Path

    def find_repo_root(start: Path) -> Path:
        for cand in [start, *start.parents]:
            if (cand / ".git").exists() and (cand / "pc2beam").is_dir():
                return cand
        raise FileNotFoundError(f"Could not locate repo root from {start}")

    project_root = find_repo_root(Path.cwd().resolve())
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

ifc_path = project_root / "data" / "model_0_z_up.ifc"
scanners_candidates = [
    project_root / "tools" / "helios_generation" / "config" / "scanners_example.yaml",
    project_root / "config" / "scanners_example.yaml",
]
scanners_yaml = next((p for p in scanners_candidates if p.exists()), scanners_candidates[0])

print("project_root:", project_root)
print("IFC:", ifc_path)
print("scanners:", scanners_yaml)

environment: local (expected Conda env: pc2beam, Python 3.12)
IFC: /Users/fnoi/Code/pc2beam/data/model_0_z_up.ifc
scanners: /Users/fnoi/Code/pc2beam/config/scanners_example.yaml


## imports

In [2]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from omegaconf import OmegaConf

from pc2beam.helios_pipeline import run_ifc_helios_pipeline
from pc2beam.helios_runner import resolve_helios_data_root
from pc2beam.ifc_io import iter_beam_records
from pc2beam.ifc_mesh_export import load_mesh_for_preview
from pc2beam.ifc_mesh_export import instance_id_for_las_hit_object_id
from pc2beam.las_io import find_first_las, las_dimension_names, read_las_scan
from pc2beam.viz import plot_point_cloud, plot_scanners_and_mesh

No stream support: No module named 'lark'


## beam metadata summary (IFC)

In [3]:
records = iter_beam_records(ifc_path)
print(f"IfcBeam count: {len(records)}")
profiles = {}
for r in records:
    key = (r.profile_name, r.material_name)
    profiles[key] = profiles.get(key, 0) + 1
print("profile / material → beam count (first 15 keys):")
for i, (k, c) in enumerate(sorted(profiles.items(), key=lambda x: -x[1])):
    if i >= 15:
        break
    print(f"  {k[0]!r} + {k[1]!r}: {c}")

IfcBeam count: 80
profile / material → beam count (first 15 keys):
  'IPE 240' + 'S235 | EN 10025-2:2004-11': 30
  'HE 140 A' + 'S235 | EN 10025-2:2004-11': 14
  'HE 260 A' + 'S235 | EN 10025-2:2004-11': 10
  'HE 100 A' + 'S235 | EN 10025-2:2004-11': 10
  'IPE 140' + 'S235 | EN 10025-2:2004-11': 9
  'HE 180 A' + 'S235 | EN 10025-2:2004-11': 4
  'HE 240 A' + 'S235 | EN 10025-2:2004-11': 2
  'HE 200 A' + 'S235 | EN 10025-2:2004-11': 1


## build HELIOS++ asset bundle (OBJ + XML)

Set `RUN_HELIOS = True` when the `helios` executable is on `PATH` and `resolve_helios_data_root()` succeeds (Conda env from `environment.yml`, or `HELIOS_DATA_PATH` for a custom install).

In [4]:
RUN_HELIOS = True

try:
    print("HELIOS data:", resolve_helios_data_root())
    helios_ready = True
except FileNotFoundError as e:
    print(e)
    helios_ready = False

if RUN_HELIOS and not helios_ready:
    RUN_HELIOS = False
    print("Disabling RUN_HELIOS: HELIOS++ data directory not found.")

result = run_ifc_helios_pipeline(
    ifc_path=ifc_path,
    scanners_yaml=scanners_yaml,
    output_root=project_root / "output" / "helios_pc2beam",
    run_simulation=RUN_HELIOS,
    helios_data_path=None,
    helios_one_obj_per_instance=True,
    mesher_linear_deflection=0.02,
)

print("run_dir:", result["run_dir"])
print("survey_xml:", result["survey_xml"])
print("ground_truth_yaml:", result.get("ground_truth_yaml"))
print("helios_one_obj_per_instance:", result.get("helios_one_obj_per_instance"))
print("scene_obj (merged):", result.get("scene_obj"))
print("instances_dir:", result.get("instances_dir"))
print("meshed beams:", result["sidecar"].meshed_beam_count, "/", result["sidecar"].beam_count)
print("pc2beam_input_txt:", result.get("pc2beam_input_txt"))
if result.get("returncode") is not None:
    print("helios return code:", result["returncode"])

HELIOS data: /opt/miniconda3/envs/pc2beam/lib/python3.12/site-packages/pyhelios
HELIOS++ VERSION 2.1.0

CWD: "/Users/fnoi/Code/pc2beam/notebooks"
seed: AUTO
surveyPath: "/Users/fnoi/Code/pc2beam/output/helios_pc2beam/e1ab02597526/surveys/pc2beam_survey.xml"
assetsPath: ["/opt/miniconda3/envs/pc2beam/lib/python3.12/site-packages/pyhelios", "/Users/fnoi/Code/pc2beam/output/helios_pc2beam/e1ab02597526", "/Users/fnoi/Code/pc2beam/notebooks", "/opt/miniconda3/envs/pc2beam/lib/python3.12/site-packages/pyhelios", "/opt/miniconda3/envs/pc2beam/lib/python3.12/site-packages/pyhelios/data", "assets/", ]
outputPath: "/Users/fnoi/Code/pc2beam/output/helios_pc2beam/e1ab02597526/sim_output/"
writeWaveform: 0
writePulse: 0
calcEchowidth: 0
fullWaveNoise: 0
splitByChannel: 0
parallelization: 1
njobs: 0
chunkSize: 32
warehouseFactor: 4
platformNoiseDisabled: 0
legNoiseDisabled: 0
rebuildScene: 0
writeScene: 1
lasOutput: 1
las10: 0
fixedIncidenceAngle: 0
gpsStartTime: 
kdtType: 4
kdtJobs: 0
kdtGeomJobs: 

In [ ]:
# Fail fast when simulation was requested but artifacts are missing.
if RUN_HELIOS:
    if result.get("returncode") not in (None, 0):
        raise RuntimeError(f"HELIOS failed with return code {result['returncode']}")
    if result.get("pc2beam_input_txt") is None:
        raise RuntimeError(
            "Simulation completed without generating pc2beam input TXT. "
            f"Inspect sim_output_dir: {result['sim_output_dir']}"
        )

## output artifacts explained (with examples)

Each simulation run writes three artifact groups under `result["run_dir"]`:

1. **Geometry for HELIOS++**
   - In split mode (`helios_one_obj_per_instance=True`): one OBJ per meshed IFC instance in `data/sceneparts/pc2beam/instances/`.
   - In merged mode: one `scene.obj` (or legacy `beams.obj`) plus matching MTL.

2. **Mapping from LAS `hitObjectId` to IFC beam/instance metadata**
   - Available in split mode via sidecar fields such as
     `meshed_instance_ids_in_part_order`, `instance_hit_mapping`, and `beam_instance_hit_mapping`.
   - `hitObjectId` is usually the 0-based HELIOS scene part index.

3. **Point cloud output**
   - LAS files in `sim_output/.../legXXX_points.las` when `RUN_HELIOS=True` and HELIOS++ is installed.

The next cell prints concrete examples from this run.

In [ ]:
from pathlib import Path

run_dir = Path(result["run_dir"])
sidecar = result["sidecar"]

print("=== 1) OBJ geometries ===")
instances_dir = result.get("instances_dir")
if instances_dir is not None and Path(instances_dir).exists():
    objs = sorted(Path(instances_dir).glob("*.obj"))
    print("instances_dir:", instances_dir)
    print("instance OBJ count:", len(objs))
    print("example instance OBJs:")
    for p in objs[:5]:
        print("  ", p)
else:
    for key in ("scene_obj", "beams_obj"):
        p = result.get(key)
        if p:
            print(f"{key}:", p)

print("\n=== 2) hitObjectId -> beam mapping (+ properties) ===")
mapping = list(getattr(sidecar, "beam_instance_hit_mapping", []) or [])
if mapping:
    print("beam_instance_hit_mapping rows:", len(mapping))
    print("example rows:")
    for row in mapping[:5]:
        print(
            "  part_index={helios_part_index}, instance_id={instance_id}, global_id={global_id}, "
            "profile={profile_name}, material={material_name}, name={name}".format(
                helios_part_index=row.get("helios_part_index"),
                instance_id=row.get("instance_id"),
                global_id=row.get("global_id"),
                profile_name=row.get("profile_name"),
                material_name=row.get("material_name"),
                name=row.get("name"),
            )
        )
else:
    beams = list(getattr(sidecar, "beams", []) or [])
    print("No explicit beam_instance_hit_mapping in this sidecar.")
    print("beam rows available:", len(beams))
    print("example beam rows:")
    for row in beams[:5]:
        print(
            "  instance_id={instance_id}, global_id={global_id}, profile={profile_name}, material={material_name}, name={name}".format(
                instance_id=row.get("instance_id"),
                global_id=row.get("global_id"),
                profile_name=row.get("profile_name"),
                material_name=row.get("material_name"),
                name=row.get("name"),
            )
        )

print("\n=== 3) point cloud (LAS) ===")
las_files = sorted((run_dir / "sim_output").rglob("*.las")) if (run_dir / "sim_output").exists() else []
if las_files:
    print("LAS file count:", len(las_files))
    print("example LAS files:")
    for p in las_files[:5]:
        print("  ", p)
else:
    print("No LAS files found in this run. Re-run with RUN_HELIOS=True and HELIOS++ available.")

## intermediate visualization — mesh + scanner layout

In [5]:
scan_cfg = OmegaConf.load(str(scanners_yaml))
positions = [tuple(map(float, s.position)) for s in scan_cfg.scanners]
orientations = []
labels = []
for s in scan_cfg.scanners:
    att = s.get("attitude_deg") or {}
    orientations.append(
        (float(att.get("yaw", 0)), float(att.get("pitch", 0)), float(att.get("roll", 0)))
    )
    labels.append(str(s.get("id", "scanner")))

mesh_path = result.get("beams_obj")
if mesh_path is not None:
    v, f, _ = load_mesh_for_preview(mesh_path, max_triangles=40_000)
    fig_scene = plot_scanners_and_mesh(
        v,
        f,
        positions,
        scanner_orientations_deg=orientations,
        scanner_labels=labels,
        vector_length=4.0,
        mesh_title="Tessellated IFC geometry + dummy scanner stations",
        max_triangles=35_000,
    )
    fig_scene.show(renderer="notebook_connected")
else:
    print(
        "Merged mesh preview skipped (per-instance OBJ mode). "
        "Instance meshes live under:", result.get("instances_dir")
    )

Merged mesh preview skipped (per-instance OBJ mode). Instance meshes live under: /Users/fnoi/Code/pc2beam/output/helios_pc2beam/e1ab02597526/data/sceneparts/pc2beam/instances


## simulated scan — LAS point cloud (when `RUN_HELIOS` was True)

In [ ]:
las_path = find_first_las(result["sim_output_dir"])
if las_path is None:
    print("No LAS found. Enable RUN_HELIOS after installing HELIOS++, or inspect sim_output_dir manually:", result["sim_output_dir"])
else:
    print("LAS:", las_path)
    print("LAS dimensions:", las_dimension_names(las_path))
    scan = read_las_scan(las_path)
    pts = scan.points
    print("points:", pts.shape[0])

    color_by = "uniform"
    features = None
    instances = None

    if scan.hit_object_id is not None:
        hid = np.asarray(scan.hit_object_id, dtype=np.int64)
        nu = len(np.unique(hid))
        print("hitObjectId unique values:", nu)
        if nu > 1:
            instances = hid
            color_by = "instance"
            print("sample hitObjectId -> IFC instance_id (via sidecar order):")
            for h in sorted(np.unique(hid))[:12]:
                iid = instance_id_for_las_hit_object_id(int(h), result["sidecar"])
                print(f"  part_index {h} -> instance_id {iid}")
        else:
            print(
                "hitObjectId is constant (often 0 with a single merged OBJ). "
                "Re-run pipeline with helios_one_obj_per_instance=True (default here)."
            )

    if instances is None and scan.intensity is not None:
        color_by = "s1"
        s1 = scan.intensity.astype(np.float32)
        s1 = (s1 - s1.min()) / (np.ptp(s1) + 1e-9)
        features = {"s1": s1}

    fig_scan = plot_point_cloud(
        pts,
        instances=instances,
        features=features,
        mode="points",
        color_by=color_by,
        show_vectors=False,
        title="Simulated TLS (color = hitObjectId / part index)" if instances is not None else "Simulated TLS point cloud",
        max_points=50_000,
        ortho_view=True,
    )
    fig_scan.show(renderer="notebook_connected")

LAS: /Users/fnoi/Code/pc2beam/output/helios_pc2beam/e1ab02597526/sim_output/pc2beam_ifc_tls/2026-04-13_21-32-17/leg000_points.las
LAS dimensions: ['X', 'Y', 'Z', 'intensity', 'return_number', 'number_of_returns', 'synthetic', 'key_point', 'withheld', 'overlap', 'scanner_channel', 'scan_direction_flag', 'edge_of_flight_line', 'classification', 'user_data', 'scan_angle', 'point_source_id', 'gps_time', 'echo_width', 'fullwaveIndex', 'hitObjectId', 'heliosAmplitude', 'ExtraBytes']
points: 243793
hitObjectId unique values: 90
sample hitObjectId -> IFC instance_id (via sidecar order):
  part_index 0 -> instance_id 1
  part_index 1 -> instance_id 2
  part_index 2 -> instance_id 3
  part_index 3 -> instance_id 4
  part_index 4 -> instance_id 5
  part_index 5 -> instance_id 6
  part_index 6 -> instance_id 7
  part_index 7 -> instance_id 8
  part_index 8 -> instance_id 9
  part_index 9 -> instance_id 10
  part_index 10 -> instance_id 11
  part_index 11 -> instance_id 12
